# Thimira - Season & Location Analysis

This notebook covers the season and location portion of the disease trend analysis task.

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
df = pd.read_csv(r'C:\Users\USER\Downloads\final_dataset.csv')
df['admission_date'] = pd.to_datetime(df['admission_date'], errors='coerce')

In [ ]:
def season_from_month(month):
    mapping = {12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',6:'Summer',7:'Summer',8:'Summer',9:'Fall',10:'Fall',11:'Fall'}
    return month.map(mapping)

df['month'] = df['admission_date'].dt.month
df['season'] = season_from_month(df['month'])
location_cols = [c for c in df.columns if c.startswith('location_')]
location_flags = df[location_cols].astype(bool)
df['location'] = location_flags.idxmax(axis=1).str.replace('location_', '', regex=False)
df.loc[location_flags.sum(axis=1).eq(0), 'location'] = 'Unknown'

In [ ]:
monthly = df.groupby(pd.Grouper(key='admission_date', freq='ME')).size().reset_index(name='encounters')
monthly.head()

In [ ]:
season_summary = df.groupby('season').agg(encounters=('encounter_id', 'count'), avg_length_of_stay=('length_of_stay', 'mean'), avg_patient_satisfaction=('patient_satisfaction', 'mean'), readmission_over_30_rate=('readmitted_>30', 'mean')).reindex(['Winter','Spring','Summer','Fall']).reset_index()
season_summary

In [ ]:
location_summary = df.groupby('location').agg(encounters=('encounter_id', 'count'), avg_length_of_stay=('length_of_stay', 'mean'), avg_patient_satisfaction=('patient_satisfaction', 'mean'), readmission_over_30_rate=('readmitted_>30', 'mean')).sort_values('encounters', ascending=False).reset_index()
location_summary

## Key Findings

- Seasonality is fairly even, with only a small spread between the busiest and quietest seasons.
- Location counts are also balanced, indicating the dataset is not heavily skewed toward one city.
- The same location tends to lead across most seasons, suggesting a consistently larger share of encounters there.